# Simple Neural Network using TensorFlow

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [3]:
def load_coffee_data():
    """ Creates a coffee roasting data set.
        roasting duration: 12-15 minutes is best
        temperature range: 175-260C is best
    """
    rng = np.random.default_rng(2)
    X = rng.random(400).reshape(-1,2)
    X[:,1] = X[:,1] * 4 + 11.5          # 12-15 min is best
    X[:,0] = X[:,0] * (285-150) + 150  # 350-500 F (175-260 C) is best
    Y = np.zeros(len(X))
    
    i=0
    for t,d in X:
        y = -3/(260-175)*t + 21
        if (t > 175 and t < 260 and d > 12 and d < 15 and d<=y ):
            Y[i] = 1
        else:
            Y[i] = 0
        i += 1

    return (X, Y.reshape(-1,1))

In [6]:
X,Y = load_coffee_data()

### Normalize Data

In [7]:
norm_l=tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X) # learns mean, variance
Xn=norm_l(X)

#### Tile/copy our data to increase the training set size and reduce the number of training epochs.

In [8]:
Xt=np.tile(Xn,(1000,1))
Yt=np.tile(Y,(1000,1))

In [10]:
tf.random.set_seed(1234)  # applied to achieve consistent results. 🔁 In short: it ensures that you get the same results each time you run your notebook, which is super helpful for debugging, tutorials, or comparing models.
model=Sequential([
    tf.keras.Input(shape=(2,)),
    Dense(3,activation='sigmoid',name='layer1'),
    Dense(1,activation='sigmoid',name='layer2')
])

In [12]:
model.summary()
# Params = (input_features × neurons) + biases

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ layer1 (Dense)                       │ (None, 3)                   │               9 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ layer2 (Dense)                       │ (None, 1)                   │               4 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 13 (52.00 B)

 Trainable params: 13 (52.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
W1,b1=model.get_layer('layer1').get_weights()
W2,b2=model.get_layer('layer2').get_weights()

print(f"W1{W1.shape}:\n", W1, f"\nb1{b1.shape}:", b1)
print(f"W2{W2.shape}:\n", W2, f"\nb2{b2.shape}:", b2)

# The weights should be of size (number of features in input, number of units in the layer) while the bias size should match the number of units in the layer:

W1(2, 3):
 [[-0.5933255   0.5532712   0.6998732 ]
 [-0.31884152 -0.769894    0.9723098 ]] 
b1(3,): [0. 0. 0.]
W2(3, 1):
 [[ 0.8448709 ]
 [-0.8067298 ]
 [ 0.82481897]] 
b2(1,): [0.]


In [16]:
# see this cell in the future labs (ignore for now)
model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
)

model.fit(
    Xt,Yt,            
    epochs=10,
)

Epoch 1/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - loss: 0.1883
Epoch 2/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 0.1149
Epoch 3/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 0.0331
Epoch 4/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 0.0151
Epoch 5/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 0.0096
Epoch 6/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - loss: 0.0066
Epoch 7/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - loss: 0.0045
Epoch 8/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 0.0031
Epoch 9/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 0.0022
Epoch 10/10
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - loss: 0.0016


#### Updated weights

##### After fitting, the weights have been updated:

In [17]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

W1:
 [[  0.17042948  14.540226   -11.141442  ]
 [ 10.425661    12.144955    -0.25549647]] 
b1: [ 12.533613    2.0057378 -12.017155 ]
W2:
 [[ 45.06192 ]
 [-47.21183 ]
 [-54.785694]] 
b2: [-14.0716]


In [18]:
X_test = np.array([
    [200,13.9],  # postive example
    [200,17]])   # negative example
x_testn=norm_l(X_test)
prediction=model.predict(x_testn)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step
[[9.8560393e-01]
 [8.7786866e-08]]


In [19]:
yhat=np.zeros_like(prediction)
for i in range(2):
    if prediction[i]>=0.5:
        yhat[i]=1
    else:
        yhat[i]=0
print(yhat)

[[1.]
 [0.]]
